In [15]:
with open("TrainingNames.txt", "r") as f:
    content = f.read()

names = [name.strip().lower() for name in content.split(',') if name.strip()]

print("Total names:", len(names))

Total names: 1000


In [20]:
chars = set()
for name in names:
    chars.update(list(name))

chars = sorted(list(chars))
chars = ['<PAD>', '<SOS>', '<EOS>'] + chars

char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for c, i in char2idx.items()}

vocab_size = len(chars)
print("Vocab size:", vocab_size)

Vocab size: 28


In [21]:
def encode(name):
    return [char2idx['<SOS>']] + [char2idx[c] for c in name] + [char2idx['<EOS>']]

encoded_names = [encode(name) for name in names]

In [22]:
encoded_names[:10]

[[1, 24, 3, 20, 3, 16, 3, 2],
 [1, 22, 3, 22, 26, 3, 2],
 [1, 21, 23, 21, 10, 11, 14, 2],
 [1, 3, 16, 11, 14, 2],
 [1, 13, 3, 14, 11, 2],
 [1, 6, 11, 24, 26, 3, 2],
 [1, 21, 3, 15, 11, 20, 3, 16, 2],
 [1, 21, 3, 16, 11, 21, 10, 2],
 [1, 15, 3, 11, 16, 3, 2],
 [1, 23, 18, 3, 21, 3, 16, 3, 2]]

In [23]:
max_len = max(len(seq) for seq in encoded_names)

def pad(seq):
    return seq + [char2idx['<PAD>']] * (max_len - len(seq))

padded_data = [pad(seq) for seq in encoded_names]

In [24]:
padded_data[:10]

[[1, 24, 3, 20, 3, 16, 3, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 22, 3, 22, 26, 3, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 21, 23, 21, 10, 11, 14, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 3, 16, 11, 14, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 13, 3, 14, 11, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 6, 11, 24, 26, 3, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 21, 3, 15, 11, 20, 3, 16, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 21, 3, 16, 11, 21, 10, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 15, 3, 11, 16, 3, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 23, 18, 3, 21, 3, 16, 3, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0]]

In [25]:
import torch
from torch.utils.data import Dataset, DataLoader

class NameDataset(Dataset):
    def __init__(self, data):
        self.data = torch.tensor(data, dtype=torch.long)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        seq = self.data[idx]
        return seq[:-1], seq[1:]   # input, target


dataset = NameDataset(padded_data)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [27]:
dataset[0]

(tensor([ 1, 24,  3, 20,  3, 16,  3,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0]),
 tensor([24,  3, 20,  3, 16,  3,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0]))

# RNN

In [228]:
embedding_dim = 64
hidden_size = 128
learning_rate = 0.001
epochs = 100

In [229]:
import torch
import torch.nn as nn

class VanillaRNNFromScratch(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()

        self.hidden_size = hidden_size

        # Embedding
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # RNN weights (manual)
        self.W_x = nn.Linear(embedding_dim, hidden_size)
        self.W_h = nn.Linear(hidden_size, hidden_size)

        # Output layer
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        batch_size, seq_len = x.size()

        x = self.embedding(x)

        # initialize hidden state
        h = torch.zeros(batch_size, self.hidden_size)

        outputs = []

        for t in range(seq_len):
            x_t = x[:, t, :]

            # RNN step
            h = torch.tanh(self.W_x(x_t) + self.W_h(h))

            # output
            out = self.fc(h)
            outputs.append(out.unsqueeze(1))

        outputs = torch.cat(outputs, dim=1)

        return outputs

In [230]:
model_rnn = VanillaRNNFromScratch(vocab_size, embedding_dim, hidden_size)

criterion = nn.CrossEntropyLoss(ignore_index=char2idx['<PAD>'])
optimizer = torch.optim.Adam(model_rnn.parameters(), lr=learning_rate)

for epoch in range(epochs):
    total_loss = 0

    for x, y in dataloader:
        outputs = model_rnn(x)

        loss = criterion(
            outputs.reshape(-1, vocab_size),
            y.reshape(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

Epoch 1: Loss = 85.7886
Epoch 2: Loss = 71.7261
Epoch 3: Loss = 67.9921
Epoch 4: Loss = 66.0862
Epoch 5: Loss = 65.0483
Epoch 6: Loss = 63.7760
Epoch 7: Loss = 63.0328
Epoch 8: Loss = 62.1136
Epoch 9: Loss = 61.6949
Epoch 10: Loss = 60.8467
Epoch 11: Loss = 60.3607
Epoch 12: Loss = 59.9159
Epoch 13: Loss = 59.3711
Epoch 14: Loss = 59.0167
Epoch 15: Loss = 58.3262
Epoch 16: Loss = 57.8918
Epoch 17: Loss = 57.5187
Epoch 18: Loss = 56.6585
Epoch 19: Loss = 56.3201
Epoch 20: Loss = 55.9332
Epoch 21: Loss = 55.3085
Epoch 22: Loss = 54.9527
Epoch 23: Loss = 54.5854
Epoch 24: Loss = 54.0451
Epoch 25: Loss = 53.7222
Epoch 26: Loss = 53.2715
Epoch 27: Loss = 52.8253
Epoch 28: Loss = 52.4986
Epoch 29: Loss = 52.0754
Epoch 30: Loss = 51.3712
Epoch 31: Loss = 51.1972
Epoch 32: Loss = 50.7134
Epoch 33: Loss = 50.4436
Epoch 34: Loss = 49.9119
Epoch 35: Loss = 49.5629
Epoch 36: Loss = 49.0955
Epoch 37: Loss = 49.0155
Epoch 38: Loss = 48.3687
Epoch 39: Loss = 47.9726
Epoch 40: Loss = 47.6414
Epoch 41:

In [231]:
def generate_name(model, max_len=20):
    model.eval()

    x = torch.tensor([[char2idx['<SOS>']]])
    name = ""

    for _ in range(max_len):
        out = model(x)
        probs = torch.softmax(out[0, -1], dim=0)

        idx = torch.multinomial(probs, 1).item()

        if idx == char2idx['<EOS>']:
            break

        name += idx2char[idx]
        x = torch.cat([x, torch.tensor([[idx]])], dim=1)

    return name

In [232]:
for _ in range(10):
    print(generate_name(model_rnn))

zarika
vindhya
nilambari
hamindi
pavan
niti
ima
ranavar
lakshmishi
darshini


In [233]:
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

model_rnn = VanillaRNNFromScratch(vocab_size, embedding_dim, hidden_size)
print("Vanilla RNN Parameters:", count_params(model_rnn))

Vanilla RNN Parameters: 30236


# BLSTM

In [197]:
embedding_dim = 32
hidden_size = 128
# num_layers =2
learning_rate = 0.001
epochs = 10

In [198]:
import torch
import torch.nn as nn

class BLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False
        )

        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)

        # out shape: (batch_size, seq_len, hidden_size)
        out, _ = self.lstm(x)

        out = self.fc(out)

        return out

In [199]:
model_blstm = BLSTM(vocab_size, embedding_dim, hidden_size, num_layers)

criterion = nn.CrossEntropyLoss(ignore_index=char2idx['<PAD>'])
optimizer = torch.optim.Adam(model_blstm.parameters(), lr=learning_rate)

for epoch in range(epochs):
    total_loss = 0

    for x, y in dataloader:
        outputs = model_blstm(x)

        loss = criterion(
            outputs.reshape(-1, vocab_size),
            y.reshape(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

Epoch 1: Loss = 95.2159
Epoch 2: Loss = 83.5286
Epoch 3: Loss = 77.8638
Epoch 4: Loss = 72.2943
Epoch 5: Loss = 69.5446
Epoch 6: Loss = 68.1160
Epoch 7: Loss = 66.9854
Epoch 8: Loss = 66.3364
Epoch 9: Loss = 65.3912
Epoch 10: Loss = 64.6075


In [200]:
for _ in range(10):
    print(generate_name(model_blstm))

simana
narrash
drigavi
kera
ooma
anaji
trishnaya
ok
trahhil
ogal


In [196]:
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("BLSTM Parameters:", count_params(model_blstm))

BLSTM Parameters: 219548


# RNN with Basic Attention Mechanism

In [148]:
embedding_dim = 16
hidden_size = 128
learning_rate = 0.001
epochs = 10

In [149]:
class RNNAttentionFromScratch(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()

        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # RNN weights
        self.W_x = nn.Linear(embedding_dim, hidden_size)
        self.W_h = nn.Linear(hidden_size, hidden_size)

        # Attention
        self.W_a = nn.Linear(hidden_size, hidden_size)

        # Output
        self.fc = nn.Linear(hidden_size * 2, vocab_size)

    def forward(self, x):
        batch_size, seq_len = x.size()
        x = self.embedding(x)

        device = x.device
        h = torch.zeros(batch_size, self.hidden_size).to(device)

        hidden_states = []

        # -------- RNN --------
        for t in range(seq_len):
            x_t = x[:, t, :]
            h = torch.tanh(self.W_x(x_t) + self.W_h(h))
            hidden_states.append(h.unsqueeze(1))

        H = torch.cat(hidden_states, dim=1)  # (B, T, H)

        outputs = []

        # -------- ATTENTION PER TIMESTEP --------
        for t in range(seq_len):
            h_t = H[:, t, :].unsqueeze(1)  # (B, 1, H)

            # score against all hidden states
            scores = torch.bmm(
                self.W_a(H),              # (B, T, H)
                h_t.transpose(1, 2)       # (B, H, 1)
            )                             # (B, T, 1)

            # ---------------------------------------------------------
            # THE FIX: CAUSAL MASK
            # Mask out future tokens by setting their scores to -infinity
            # ---------------------------------------------------------
            scores[:, t+1:, :] = float('-inf')

            weights = torch.softmax(scores, dim=1)

            context = torch.sum(weights * H, dim=1)  # (B, H)

            # combine current + context
            combined = torch.cat([h_t.squeeze(1), context], dim=1)

            out = self.fc(combined)  # (B, V)
            outputs.append(out.unsqueeze(1))

        outputs = torch.cat(outputs, dim=1)

        return outputs

In [150]:
model_rnn_attention = RNNAttentionFromScratch(vocab_size, embedding_dim, hidden_size)

criterion = nn.CrossEntropyLoss(ignore_index=char2idx['<PAD>'])
optimizer = torch.optim.Adam(model_rnn_attention.parameters(), lr=learning_rate)

for epoch in range(epochs):
    total_loss = 0

    for x, y in dataloader:
        outputs = model_rnn_attention(x)

        loss = criterion(
            outputs.reshape(-1, vocab_size),
            y.reshape(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

Epoch 1: Loss = 89.1084
Epoch 2: Loss = 74.9577
Epoch 3: Loss = 70.9459
Epoch 4: Loss = 68.3632
Epoch 5: Loss = 67.1376
Epoch 6: Loss = 66.0032
Epoch 7: Loss = 65.0137
Epoch 8: Loss = 64.0493
Epoch 9: Loss = 63.2384
Epoch 10: Loss = 62.3295


In [151]:
for _ in range(10):
    print(generate_name(model_rnn_attention))

soilial
ga
bhiamrn
vitar
suraj
giral
janjyi
udubi
viyhavika
maladhan


In [152]:
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("RNN with Attention Parameters:", count_params(model_rnn_attention))

RNN with Attention Parameters: 42844


# Task-2

In [223]:
def generate_batch(model, n=500):
    generated = []
    for _ in range(n):
        name = generate_name(model)
        if name:  # ignore empty
            generated.append(name)
    return generated

def compute_novelty(generated, train_set):
    novel = [name for name in generated if name not in train_set]
    return len(novel) / len(generated)

def compute_diversity(generated):
    return len(set(generated)) / len(generated)

def evaluate_model(model, name):
    generated = generate_batch(model, n=500)

    novelty = compute_novelty(generated, names)
    diversity = compute_diversity(generated)

    print(f"\n{name} Results:")
    print(f"Novelty Rate: {novelty:.4f}")
    print(f"Diversity: {diversity:.4f}")

    return novelty, diversity

In [224]:
rnn_nov, rnn_div = evaluate_model(model_rnn, "Vanilla RNN")


Vanilla RNN Results:
Novelty Rate: 1.0000
Diversity: 0.9895


In [ ]:
rnn_nov, rnn_div = evaluate_model(model_blstm, "BLSTM")


Vanilla RNN Results:
Novelty Rate: 1.0000
Diversity: 1.0000


In [227]:
attn_nov, attn_div = evaluate_model(model_rnn_attention, "RNN + Attention")


RNN + Attention Results:
Novelty Rate: 0.9900
Diversity: 0.9980
